<div style="display:flex; align-items:center; gap:18px; text-align:left">
<img src="https://sebastiancontz.github.io/ust-introduccion-machine-learning/assets/logo-ust.svg" width="100" alt="Logo de la Universidad Santo Tomás">
<div>
<p>Ingeniería en Información y Control de Gestión</p>
<p>Facultad de Economía y Negocios</p>
<p>Introducción a Machine Learning</p>
<p>Semana 04: Preparación de datos</p>
</div>
</div>

# 04 · Preparación de datos

La clase pasada **detectamos y documentamos**. Hoy **decidimos y corregimos**.

El archivo es un año de facturación de un mayorista de artículos de regalo: **541.910 registros** en
25.900 facturas. Cada registro es **una línea de factura** —un producto dentro de una factura—, así que
una misma factura ocupa varios registros. No es una venta completa, y no es un cliente.

## Qué queda al terminar

Dos cosas, y una sin la otra no sirve:

1. el **archivo corregido**;
2. la **bitácora** que justifica cada decisión.

El archivo solo dice *qué* quedó distinto. La bitácora dice **por qué**, y es lo único que permite
que otra persona —o ustedes mismos en tres meses— entiendan las decisiones. Lo difícil de hoy no es
la técnica: es sostener el **porqué** de cada una.

## Cómo se trabaja

Siete decisiones en tres tiempos: **dos** las toma el docente en pantalla, **tres** se resuelven
entre todos, y **dos** quedan para ustedes.

Las siete cuentan. La celda final del cuaderno **no exporta nada mientras falte alguna**: un archivo
corregido a medias, sin las decisiones que lo explican, es exactamente lo que esta clase enseña a no
producir.

> **Datos reales**, no un ejemplo armado para practicar. La atribución completa está al final del
> cuaderno.

## El árbol de decisión

Las tres preguntas de la clase, que son lo único de hoy que sirve para un archivo que no es este.

<p align="center"><img src="" width="980" alt="Árbol de decisión de limpieza. Tres preguntas en cadena: uno, cómo se generó este dato, quién lo escribió, cuándo y con qué sistema; dos, si estará el dato el día de la decisión o se llena recién después del hecho; y tres, qué efecto tiene cada opción. De la segunda pregunta sale una rama roja rotulada no, hacia una caja aparte: se excluye y se documenta, no se discute estrategia. De la tercera cuelgan tres opciones: excluir, que pierde filas y segmentos; imputar, cuyas versiones simples aplanan la variabilidad; y marcar, que conserva la señal del vacío y suma una columna. Al pie: ninguna es gratis, la decisión es cuál costo se acepta y por qué; y las tres preguntas se responden con conocimiento del proceso, no con estadística"></p>

## Preparación del entorno

En Colab estas librerías ya vienen instaladas, así que la celda termina en segundos. Está igual
porque dejar escrito de qué depende un análisis es parte de que sea reproducible.

In [1]:
%%capture
!pip install -q pandas numpy pyarrow scikit-learn

## Cargar datos

El archivo viene en **Parquet**, que guarda el tipo de cada columna. Eso elimina de entrada toda una
familia de problemas —adivinar separadores, formatos de fecha, qué texto representa un vacío— y deja
a la vista la familia que importa hoy: la del **dato mismo**. Se lee con **`read_parquet`**, que es a
Parquet lo que `read_csv` es a un CSV.

Un gesto que va a repetirse toda la clase: una **condición** no devuelve un dato, devuelve una
**máscara** —True o False por fila—, y usarla entre corchetes deja solo lo que cumple.

In [2]:
import os
import pandas as pd

REPO = 'https://raw.githubusercontent.com/sebastiancontz/ust-introduccion-machine-learning-colab/main/ediciones/2026/datasets/'
BASE = '../datasets/' if os.path.exists('../datasets') else REPO

In [3]:
ventas = pd.read_parquet(BASE + 'ventas_online.parquet')   # como read_csv, pero trae los tipos

ventas.head()

,n_factura,codigo_producto,descripcion_producto,cantidad,fecha_factura,precio_unitario,id_cliente,pais
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


`info()` es la primera lectura del archivo: cuántos registros hay, qué tipo tiene cada columna y cuántos
valores no vacíos trae. Las ocho columnas de acá abajo son las mismas que están en las slides.

In [4]:
ventas.info()

<class 'pandas.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   n_factura             541910 non-null  str           
 1   codigo_producto       541910 non-null  str           
 2   descripcion_producto  540456 non-null  str           
 3   cantidad              541910 non-null  int64         
 4   fecha_factura         541910 non-null  datetime64[us]
 5   precio_unitario       541910 non-null  float64       
 6   id_cliente            406830 non-null  float64       
 7   pais                  541910 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 59.5 MB


In [5]:
print('Registros:', len(ventas), '· Facturas:', ventas['n_factura'].nunique())

Registros: 541910 · Facturas: 25900


In [6]:
vacias = ventas.isna().sum()

print('Celdas vacías por columna:')
print(vacias[vacias > 0])   # la máscara deja solo las columnas que sí tienen vacíos

Celdas vacías por columna:
descripcion_producto      1454
id_cliente              135080
dtype: int64


Seis de las ocho columnas no tienen una sola celda vacía. Los vacíos están en dos, y una está
**dentro** de la otra: eso se vio en clase y se decide más abajo.

## La bitácora de limpieza

Es la bitácora de la clase 3 con tres campos nuevos. Allá se documentó **qué se observó**; acá,
**qué se decidió hacer**.

La función pide cinco campos, y ninguno es burocracia: sin `proceso generador` no se puede
justificar nada, y sin `efecto esperado` no se puede auditar la decisión después.

**«Se imputó con la mediana» no es una justificación.** Es la técnica. La justificación dice por qué
esa decisión y no otra, en términos del negocio.

Una convención del cuaderno, para que no sorprenda: **cada decisión ocupa dos celdas**, una que
limpia y otra que anota. Son dos actos distintos, y separarlos tiene una ventaja práctica — volver a
ejecutar la limpieza las veces que haga falta no duplica la línea de bitácora.

In [7]:
DECISIONES_ESPERADAS = 7   # las siete de la clase; la celda final revisa que estén todas

BITACORA = []

In [8]:
def anotar(variable, proceso_generador, decision, justificacion, efecto_esperado):
    """Agrega una decisión a la bitácora y la devuelve para revisarla.

    variable          : la columna afectada
    proceso_generador : por qué ocurre, según el negocio. Si no se sabe, decirlo así
    decision          : qué se hizo
    justificacion     : por qué esa y no otra. La técnica NO es una justificación
    efecto_esperado   : qué cambia en los datos y en quién queda representado
    """
    BITACORA.append({'variable': variable,
                     'proceso generador': proceso_generador, 'decisión': decision,
                     'justificación': justificacion, 'efecto esperado': efecto_esperado})
    return pd.DataFrame(BITACORA[-1:])

Y una copia del archivo para trabajar, con **`copy()`**. **El original no se toca**: si una decisión
sale mal, hay que poder volver.

In [9]:
limpio = ventas.copy()
print('Copia de trabajo:', limpio.shape)

Copia de trabajo: (541910, 8)


---

# Parte 1 · Miren esto conmigo

Dos decisiones con el ciclo completo: qué se encontró, por qué ocurre, qué se decide, por qué esa y
no otra, y qué cambia.

## Decisión 1 · Las descripciones vacías

`descripcion_producto` tiene celdas vacías. Antes de decidir, hay que ver **de qué tamaño** es el
problema y **dónde** está.

In [10]:
sin_desc = limpio['descripcion_producto'].isna()

# .sum() sobre una máscara cuenta los True, porque False vale 0 y True vale 1
print('Registros sin descripción:', sin_desc.sum())
print(f'Porcentaje del archivo: {100 * sin_desc.mean():.2f} %')

Registros sin descripción: 1454
Porcentaje del archivo: 0.27 %


In [11]:
print('De esos registros:')

# .str agrupa las operaciones de texto de una columna: acá, si empieza con 'C'
pd.DataFrame({
    'registros': [
        (sin_desc & limpio['id_cliente'].isna()).sum(),
        (sin_desc & (limpio['precio_unitario'] == 0)).sum(),
        (sin_desc & limpio['n_factura'].str.startswith('C')).sum(),
    ],
}, index=['sin cliente', 'con precio cero', 'con prefijo C'])   # index= nombra las filas

De esos registros:


,registros
sin cliente,1454
con precio cero,1454
con prefijo C,0


Dos cifras coinciden con el total y la tercera es cero: **los 1.454 registros sin descripción no tienen
cliente, tienen precio cero, y ninguna lleva prefijo de cancelación**. Una firma que se repite así en tres columnas no
es azar; es compatible con un proceso común, aunque confirmarlo le toca a quien conoce el dato.

Ahora, ¿se pueden recuperar? El `codigo_producto` de esos registros aparece en otros que sí traen la
descripción, así que el código podría dar el nombre.

**Pero antes de rellenar hay que preguntarse si el código responde una sola cosa.** La celda siguiente
cuenta, para cada código, cuántos nombres distintos se le conocen, y usa **`map()`** para traducir
cada código por ese recuento.

In [12]:
# cuántos nombres DISTINTOS conoce cada código, contando solo los registros que sí traen descripción
nombres_por_codigo = (limpio.loc[~sin_desc]
                      .groupby('codigo_producto')['descripcion_producto'].nunique())

# para cada registro vacío: cuántos nombres conoce su código (vacío = el código no está en ninguno)
conocidos = limpio.loc[sin_desc, 'codigo_producto'].map(nombres_por_codigo)

In [13]:
pd.DataFrame({
    'registros': [
        (conocidos == 1).sum(),
        (conocidos > 1).sum(),
        conocidos.isna().sum(),
    ],
}, index=['código con UN nombre conocido', 'código con VARIOS nombres', 'código sin catálogo'])

,registros
código con UN nombre conocido,1033
código con VARIOS nombres,309
código sin catálogo,112


Solo el primer grupo se puede completar sin inventar nada. Los otros dos, no — y conviene ver por qué.

In [14]:
# los códigos ambiguos que aparecen entre los registros vacíos, con todos sus nombres conocidos
ambiguos = limpio.loc[sin_desc & (conocidos > 1), 'codigo_producto'].unique()

print('Códigos ambiguos entre los registros a completar:', len(ambiguos))

Códigos ambiguos entre los registros a completar: 174


In [15]:
for cod in sorted(ambiguos)[:3]:
    nombres = sorted(limpio.loc[~sin_desc & (limpio['codigo_producto'] == cod),
                                'descripcion_producto'].unique())
    print(f'  {cod}: {nombres}')

  10080: ['GROOVY CACTUS INFLATABLE', 'check']
  15058A: ['BLUE POLKADOT GARDEN PARASOL', 'wet/rusty']
  16045: ['POPART WOODEN PENCILS ASST', 'check']


Ahí está el problema, y la salida lo muestra. Al código **10080** se le conocen dos nombres:
`GROOVY CACTUS INFLATABLE` y `check`. Uno es el producto; el otro es una **anotación del bodeguero**
escrita en el campo equivocado — el mismo defecto que se decide más abajo, en la Decisión 4.

Tomar «el primero que aparezca» habría rellenado 309 registros con lo que hubiera quedado antes en el
archivo, que en varios casos es la anotación y no el producto. **El orden de las filas no es un
criterio de negocio.**

Así que se completa **solo lo inequívoco**: los códigos con un único nombre conocido. El resto queda
**sin completar**, y eso es una respuesta honesta y no un vacío olvidado: su código no da un nombre
único, así que rellenarlo exigiría una decisión que no se puede tomar desde el archivo.

Esto **no es imputación estadística**. No se rellena con la moda ni con un promedio: se completa con
una **regla de negocio** —el código manda, cuando el código dice una sola cosa—, que es información
real y no una estimación.

In [16]:
# el catálogo se arma SOLO con los códigos que tienen un único nombre conocido
codigos_inequivocos = nombres_por_codigo[nombres_por_codigo == 1].index
catalogo = (limpio.loc[~sin_desc & limpio['codigo_producto'].isin(codigos_inequivocos)]
            .groupby('codigo_producto')['descripcion_producto'].first())

In [17]:
# map() traduce cada código por su nombre del catálogo; fillna() lo usa SOLO donde faltaba
limpio['descripcion_producto'] = limpio['descripcion_producto'].fillna(
    limpio['codigo_producto'].map(catalogo)
)

In [18]:
completadas = sin_desc.sum() - limpio['descripcion_producto'].isna().sum()

print('Registros completados desde el catálogo:', completadas)
print('Registros que siguen sin descripción :', limpio['descripcion_producto'].isna().sum())
print('  (de código ambiguo o sin catálogo: requieren revisión o una decisión documentada)')

Registros completados desde el catálogo: 1033
Registros que siguen sin descripción : 421
  (de código ambiguo o sin catálogo: requieren revisión o una decisión documentada)


In [19]:
anotar(
    variable='descripcion_producto',
    proceso_generador='registros que no son ventas y se cargaron sin nombre de producto',
    decision=f'completar SOLO desde códigos con un único nombre conocido: {completadas} registros. Los códigos con varios nombres no se rellenan',
    justificacion='cuando el código conoce un solo nombre, ese nombre es información real; cuando conoce varios, elegir por orden de aparición no es un criterio de negocio y arriesga cargar una anotación del bodeguero como si fuera el producto',
    efecto_esperado=f'ninguna fila se pierde; quedan {limpio["descripcion_producto"].isna().sum()} sin descripción, sin completar a ciegas: su código no da un nombre único y la decisión queda pendiente',
)

,variable,proceso generador,decisión,justificación,efecto esperado
0,descripcion_producto,registros que no son ventas y se cargaron sin ...,completar SOLO desde códigos con un único nomb...,"cuando el código conoce un solo nombre, ese no...",ninguna fila se pierde; quedan 421 sin descrip...


## Decisión 2 · Una columna que codifica dos hechos

`n_factura` guarda el número del documento **y**, con el prefijo `C`, si la venta fue cancelada. Dos
hechos en una columna no se pueden contar por separado.

In [20]:
es_cancelacion = limpio['n_factura'].str.startswith('C')

print('Líneas con prefijo C:', es_cancelacion.sum())
print('De esas, con cantidad positiva:', (es_cancelacion & (limpio['cantidad'] > 0)).sum())

Líneas con prefijo C: 9288
De esas, con cantidad positiva: 0


Cero excepciones: **toda línea cancelada tiene cantidad negativa**. La regla se cumple en ese
sentido, y conviene anotarlo — una auditoría también documenta lo que **sí** está bien.

Separar los dos hechos en dos columnas no pierde nada y habilita contar cada uno.

In [21]:
limpio['cancelada'] = es_cancelacion

# removeprefix('C') quita UNA 'C' inicial. No usar lstrip('C'): recibe un conjunto de caracteres,
# no un prefijo, y con 'CC12345' devolvería '12345' en vez de 'C12345'
limpio['n_documento'] = limpio['n_factura'].str.removeprefix('C')

print(limpio[['n_factura', 'n_documento', 'cancelada']].head(3).to_string(index=False))

n_factura n_documento  cancelada
   536365      536365      False
   536365      536365      False
   536365      536365      False

**Una advertencia sobre `cancelada`, que depende de para qué se use.**

Acá es una corrección de formato: separa dos hechos que venían pegados en una columna. Pero si el
problema fuera **anticipar qué pedidos se van a cancelar**, esta columna —y la cantidad negativa que
la acompaña— **serían la respuesta escrita de otra forma**: se llenan cuando la cancelación ya
ocurrió, no antes.

La pregunta de la clase 3 lo detecta: **¿con información de qué momento se llena este campo?** Una
columna que solo existe después del hecho no puede usarse para predecirlo. Separarla está bien;
tratarla como predictora sin hacerse esa pregunta, no.

In [22]:
anotar(
    variable='n_factura',
    proceso_generador='el sistema marca la cancelación con un prefijo en vez de una columna propia',
    decision='separar en n_documento y cancelada; se conserva n_factura',
    justificacion='dos hechos en una columna no se pueden contar ni filtrar por separado',
    efecto_esperado='se puede contar cancelaciones sin parsear texto; no se pierde información. Queda registrado que cancelada es posterior al hecho y no sirve para anticiparlo',
)

,variable,proceso generador,decisión,justificación,efecto esperado
0,n_factura,el sistema marca la cancelación con un prefijo...,separar en n_documento y cancelada; se conserv...,dos hechos en una columna no se pueden contar ...,se puede contar cancelaciones sin parsear text...


---

# Parte 2 · Lo hacemos juntos

Tres decisiones, una por tipo de problema. El código está escrito: lo que hacen ustedes es **decidir
antes de ejecutar**, y después leer la salida.

## Decisión 3 · Los duplicados

**Antes de ejecutar, decidan.** El archivo tiene miles de filas idénticas en todas sus columnas.
¿Se borran las repeticiones, se conservan, o falta información para responder? Comprométanse con una
respuesta antes de mirar la salida.

Y una precisión sobre qué se está contando, porque tres llamadas casi iguales dan tres números:

- **`duplicated()`** marca cada fila que **ya apareció antes**, así que no marca la primera de cada
  grupo. Cuenta las **repeticiones**, que es lo que se eliminaría.
- **`duplicated(keep=False)`** marca **todas** las filas de cada grupo repetido, incluida la primera:
  el **tamaño del conjunto afectado**, no cuántas se van.
- **`duplicated(subset=[...])`** compara **solo esas columnas** en vez de la fila completa.

In [23]:
print('Repeticiones (al conservar la primera de cada grupo):', limpio.duplicated().sum())
print('Filas involucradas en algún grupo repetido        :', limpio.duplicated(keep=False).sum())

Repeticiones (al conservar la primera de cada grupo): 5268


Filas involucradas en algún grupo repetido        : 10147


In [24]:
clave = ['n_factura', 'codigo_producto', 'cantidad', 'precio_unitario']
print()
print('Repeticiones usando solo la clave de negocio', clave, ':')
print(' ', limpio.duplicated(subset=clave).sum())


Repeticiones usando solo la clave de negocio ['n_factura', 'codigo_producto', 'cantidad', 'precio_unitario'] :
  5271


Dos números distintos para la misma pregunta, y la diferencia es el contenido de esta decisión.

`duplicated()` compara la **fila completa**. Con `subset` compara solo la clave de negocio, y
aparecen **más repeticiones**: filas que son el mismo hecho pero difieren en alguna columna.

Y la pregunta de fondo no la resuelve pandas: **una factura puede tener legítimamente dos líneas del
mismo producto** —dos cajas cargadas por separado—. Si eso es posible en este negocio, borrar por
clave de negocio elimina ventas reales.

In [25]:
antes = len(limpio)
limpio = limpio.drop_duplicates()   # sin subset: compara la fila completa, la opción conservadora

print('Filas antes :', antes)
print('Filas después:', len(limpio))

Filas antes : 541910
Filas después: 536642


In [26]:
anotar(
    variable='(todas)',
    proceso_generador='cargas repetidas del mismo archivo; no se distingue de una recompra en el mismo minuto',
    decision=f'eliminar las {antes - len(limpio)} repeticiones de FILA COMPLETA, conservando la primera de cada grupo',
    justificacion='una factura puede tener dos líneas del mismo producto, así que borrar por clave de negocio eliminaría ventas reales',
    efecto_esperado=f'el archivo baja a {len(limpio)} filas; los hechos repetidos dejan de pesar doble',
)

,variable,proceso generador,decisión,justificación,efecto esperado
0,(todas),cargas repetidas del mismo archivo; no se dist...,eliminar las 5268 repeticiones de FILA COMPLET...,una factura puede tener dos líneas del mismo p...,el archivo baja a 536642 filas; los hechos rep...


**Desde acá, las cifras del cuaderno y las de las slides dejan de coincidir exactamente.** Las
slides cuentan sobre el archivo original y nosotros acabamos de eliminar repeticiones, así que todo
lo que se cuente de ahora en adelante sale un poco más bajo. No es un error: **cada decisión mueve la
base de la siguiente**, y esa es una razón más para dejarlas anotadas.

## Decisión 4 · Las anotaciones escritas a mano

`descripcion_producto` no siempre trae un nombre de producto. A veces trae una **nota del
bodeguero**: son las mismas que ensuciaron el catálogo en la Decisión 1.

**Antes de ejecutar, decidan.** ¿Se arregla pasando todo a minúsculas y quitando espacios?

In [27]:
notas = ['check', 'damages', 'damaged', '?', 'Found', 'found']

# value_counts() cuenta cada valor; reindex() deja solo los de la lista, en ese orden
print(limpio['descripcion_producto'].value_counts().reindex(notas, fill_value=0))

descripcion_producto
check      159
damages     45
damaged     43
?           47
Found        8
found       25
Name: count, dtype: int64


In [28]:
print('Códigos con más de una descripción distinta:',
      (limpio.groupby('codigo_producto')['descripcion_producto'].nunique() > 1).sum(),
      'de', limpio['codigo_producto'].nunique())

Códigos con más de una descripción distinta: 650 de 4070


Acá hay **dos problemas distintos** y conviene no confundirlos.

`Found` y `found` son **la misma palabra escrita de dos formas**: eso lo resuelve normalizar a
minúsculas. Pero `check` **no es un producto**, y ninguna normalización lo va a convertir en uno.

Normalizar el texto es barato y arregla lo primero. Lo segundo necesita una decisión.

In [29]:
# strip() quita espacios de los extremos ('check ' != 'check'); lower() pasa a minúsculas
# va en una columna nueva para no perder el texto original
limpio['descripcion_norm'] = limpio['descripcion_producto'].str.strip().str.lower()

print('Descripciones distintas antes :', limpio['descripcion_producto'].nunique())
print('Descripciones distintas después:', limpio['descripcion_norm'].nunique())

Descripciones distintas antes : 4223
Descripciones distintas después: 4194


In [30]:
anotar(
    variable='descripcion_producto',
    proceso_generador='el campo del nombre se usó como cuaderno de notas cuando no había dónde anotar',
    decision='normalizar a minúsculas y sin espacios sobrantes en una columna nueva; las anotaciones no se borran',
    justificacion='normalizar unifica variantes de la misma palabra, pero no convierte una nota en un producto',
    efecto_esperado='menos categorías duplicadas; los registros anotados quedan identificables para decidirlos',
)

,variable,proceso generador,decisión,justificación,efecto esperado
0,descripcion_producto,el campo del nombre se usó como cuaderno de no...,normalizar a minúsculas y sin espacios sobrant...,normalizar unifica variantes de la misma palab...,menos categorías duplicadas; los registros ano...


## Decisión 5 · Los códigos que no son productos

`codigo_producto` guarda productos —cinco dígitos— y también cosas que no lo son.

**Antes de ejecutar, decidan.** Si un registro no corresponde a un producto, ¿se borra, se deja como
está, o se hace otra cosa con ella?

In [31]:
# str.match ancla al INICIO del texto, así que \d{5} pide cinco dígitos ahí mismo (no hace falta
# escribir ^). La virgulilla ~ invierte la máscara: quedan los que NO calzan
no_producto = ~limpio['codigo_producto'].str.match(r'\d{5}')

print(limpio.loc[no_producto, 'codigo_producto'].value_counts().head(8).to_string())
print()
print('Líneas que no son producto:', no_producto.sum())

codigo_producto
POST            1257
DOT              710
M                566
C2               144
D                 77
S                 62
BANK CHARGES      37
AMAZONFEE         34

Líneas que no son producto: 2990


`POST` es franqueo, `DOT` es un cargo por transporte, `M` es un ajuste manual, `BANK CHARGES` son
comisiones bancarias.

**No se borran.** Son hechos contables reales, solo que cargados en una tabla de ventas. Borrarlos
haría desaparecer costos que la empresa efectivamente tuvo. Se **separan**, que es distinto.

**Y la regla no es perfecta, que es justamente el punto.** Entre las marcadas hay códigos que **sí
son productos** —`DCGSSGIRL` es `GIRLS PARTY BAG`, `PADS` es `PADS TO MATCH ALL CUSHIONS`— y los
`gift_0001_*`, que son vales de regalo: **ingreso, no costo**. «Cinco dígitos» es una heurística, no
una definición del negocio. Marcar en vez de borrar deja esos casos recuperables; borrarlos habría
sido irreversible, y nadie se habría enterado.

In [32]:
limpio['es_producto'] = ~no_producto

print(limpio['es_producto'].value_counts().to_string())

es_producto
True     533652
False      2990


In [33]:
anotar(
    variable='codigo_producto',
    proceso_generador='la tabla de ventas se usa también para cargar franqueo, transporte y ajustes contables',
    decision=f'marcar las {no_producto.sum()} con la columna es_producto; no se eliminan',
    justificacion='la mayoría son hechos contables reales y borrarlos haría desaparecer costos que la empresa tuvo; además la regla de los cinco dígitos es una heurística que deja adentro algunos productos y vales de regalo, y marcar en vez de borrar los mantiene recuperables',
    efecto_esperado='un análisis de productos puede filtrarlas; la contabilidad no pierde nada',
)

,variable,proceso generador,decisión,justificación,efecto esperado
0,codigo_producto,la tabla de ventas se usa también para cargar ...,marcar las 2990 con la columna es_producto; no...,la mayoría son hechos contables reales y borra...,un análisis de productos puede filtrarlas; la ...


---

# Parte 3 · Ahora ustedes

Dos decisiones, y las dos van a la bitácora. **La primera es obligatoria.**

## Antes de decidir: ¿por qué falta?

La estrategia para un vacío no se elige por el porcentaje. Se elige por **el motivo de la ausencia**.
Tres casos, con los nombres que van a encontrar en cualquier documentación:

- **Ausencia pareja** (*MCAR*, «faltante completamente al azar»): la probabilidad de que falte es la
  misma en todas las filas.
- **Ausencia que depende de lo observado** (*MAR*, «faltante al azar»): depende de algo que el archivo
  **sí** registra en otra columna.
- **Ausencia informativa** (*MNAR*, «faltante no al azar»): depende de algo que el archivo **no**
  registra.

**Qué se puede ver en el archivo y qué no**, que es la parte que importa:

- Se puede **descartar el primero**: si los registros con vacío se comportan distinto de los demás, la
  ausencia no es pareja. Eso lo muestra la celda de abajo, con este archivo.
- **Separar los dos últimos no se mira, se pregunta.** Depende de algo que el archivo no contiene, así
  que ningún cálculo lo decide: la respuesta la tiene quien conoce el proceso que generó el dato.

*(Los tres nombres son vocabulario estándar fuera de la bibliografía del curso; se usan acá porque
van a encontrarlos escritos así.)*

## Decisión 6 · Obligatoria — el `id_cliente` que falta

Es la decisión más difícil de la clase, porque es la única que obliga a pronunciarse sobre **a quién
se deja fuera**.

**`id_cliente` es un identificador, no un predictor.** No mide nada del cliente: lo nombra. Esa
distinción decide la estrategia, y tiene una consecuencia inmediata — **imputar no está entre las
opciones**. Rellenar un identificador con el valor más frecuente inventa un cliente que no existe y
funde compras de personas distintas en una sola.

Un detalle antes de mirar las cifras: en la clase vieron **24,9 %**, y acá va a salir un poco más
alto. No es un error — ya eliminamos las repeticiones, así que el denominador cambió. **Cada decisión
mueve la base de la siguiente**, y esa es una razón más para dejarlas anotadas.

In [34]:
sin_cliente = limpio['id_cliente'].isna()

grupo_con = limpio[~sin_cliente]   # ~ invierte la máscara: las que SÍ traen cliente
grupo_sin = limpio[sin_cliente]

print(f'Registros sin cliente: {sin_cliente.sum()} ({100 * sin_cliente.mean():.1f} %)')

Registros sin cliente: 135037 (25.2 %)


In [35]:
pd.DataFrame({
    'unidades por línea': [
        grupo_con['cantidad'].mean(),
        grupo_sin['cantidad'].mean(),
    ],
    'monto de la línea (GBP)': [
        (grupo_con['cantidad'] * grupo_con['precio_unitario']).mean(),
        (grupo_sin['cantidad'] * grupo_sin['precio_unitario']).mean(),
    ],
    'líneas con precio cero': [
        (grupo_con['precio_unitario'] == 0).sum(),
        (grupo_sin['precio_unitario'] == 0).sum(),
    ],
}, index=['con cliente', 'sin cliente']).round(2)

,unidades por línea,monto de la línea (GBP),líneas con precio cero
con cliente,12.18,20.61,40
sin cliente,2.00,10.72,2470


Los dos grupos **no se comportan igual**. Con eso queda descartada la ausencia pareja: lo que falta
no falta al azar.

Las tres opciones defendibles, y ninguna es gratis:

1. **Excluir** esas filas, asumiendo por escrito a quién se deja fuera.
2. **Conservarlas y marcar** la ausencia con una columna indicadora.
3. Tratar **«sin cliente» como una categoría propia**, si resulta ser una forma de operar y no un
   error.

La celda siguiente trae **las tres escritas y listas para ejecutar**. Descomenten **una** —o escriban
la suya— y ejecútenla.

**Sobre la opción 3, un cuidado técnico:** `id_cliente` es una columna decimal (`17850.0`). Escribir
la palabra «sin cliente» dentro de ella mezclaría texto con números y rompería la columna como
identificador. La categoría va en una **columna aparte**, que es lo que hace la plantilla.

**Y lo que la justificación tiene que hacer**, además de defender la elegida: **descartar por escrito
las otras dos**, con su razón. Una decisión sin alternativas descartadas no es una decisión, es una
preferencia.

In [36]:
# SU DECISIÓN · descomenten UNA de las tres y ejecuten

# --- Opción 1 · EXCLUIR las filas sin cliente -------------------------------------------------
# limpio = limpio[~sin_cliente]

# --- Opción 2 · MARCAR la ausencia con una columna indicadora ---------------------------------
# limpio['sin_cliente'] = sin_cliente

# --- Opción 3 · «sin cliente» como CATEGORÍA PROPIA, en columna aparte -------------------------
# id_cliente queda intacto: la categoría no se escribe dentro de la columna decimal
# limpio['segmento_cliente'] = sin_cliente.map({True: 'sin cliente', False: 'identificado'})

print('Filas en el archivo de trabajo:', len(limpio))
print('Columnas:', list(limpio.columns))

Filas en el archivo de trabajo: 536642
Columnas: ['n_factura', 'codigo_producto', 'descripcion_producto', 'cantidad', 'fecha_factura', 'precio_unitario', 'id_cliente', 'pais', 'cancelada', 'n_documento', 'descripcion_norm', 'es_producto']


In [37]:
# SU LÍNEA DE BITÁCORA, en su propia celda como las cinco anteriores
# en `justificacion` descarten por escrito las dos opciones que NO tomaron, con su razón
#
# anotar(
#     variable='id_cliente',
#     proceso_generador='...',
#     decision='...',
#     justificacion='... Descarto excluir porque ... y la categoría propia porque ...',
#     efecto_esperado='...',
# )

## Decisión 7 · Libre — elijan una de las tres

Las tres están sin resolver en el archivo. Cada una viene con **la evidencia mínima para decidir** y
con **la pregunta de negocio** que hay que responder. Elijan **una**, decidan y anótenla.

La evidencia no trae la respuesta: trae con qué sostenerla.

**Un aviso si en la decisión anterior eligieron excluir.** Las tres alternativas viven casi por
completo en los registros sin cliente, así que la evidencia de abajo se va a calcular sobre lo que quedó
y va a salir mucho más chica, o vacía. No es un error: es la decisión anterior moviendo la base. Si
les pasa, **eso mismo va en la bitácora** — es el efecto de su decisión, y observarlo vale tanto como
la decisión.

### A · Cantidad negativa sin prefijo de cancelación

Las canceladas se marcan con `C` y todas tienen cantidad negativa. Pero hay registros negativos **sin**
esa marca.

**La pregunta de negocio: ¿son ventas?** Si lo son, restan del total vendido. Si no lo son, sumarlas
como ventas negativas deforma cualquier cifra de ventas.

Variables que hay que mirar: `cantidad`, `n_factura`, `id_cliente`, `precio_unitario` y lo que digan
las descripciones.

In [38]:
neg_sin_c = (~limpio['n_factura'].str.startswith('C')) & (limpio['cantidad'] < 0)

print('Registros:', neg_sin_c.sum())
print()

Registros: 1336



In [39]:
print(pd.DataFrame({
    'registros': [
        (neg_sin_c & limpio['id_cliente'].isna()).sum(),
        (neg_sin_c & (limpio['precio_unitario'] == 0)).sum(),
        (neg_sin_c & limpio['descripcion_producto'].notna()).sum(),
    ],
}, index=['sin cliente', 'con precio cero', 'con alguna descripción']).to_string())

                        registros
sin cliente                  1336
con precio cero              1336
con alguna descripción       1113


In [40]:
print('Qué dicen las que traen descripción:')
print(limpio.loc[neg_sin_c, 'descripcion_producto'].value_counts().head(6).to_string())

Qué dicen las que traen descripción:
descripcion_producto
check                    120
damages                   45
damaged                   42
?                         41
sold as set on dotcom     20
Damaged                   14


### B · Líneas con precio cero

Un precio cero puede ser un dato que falta, un regalo o una muestra, o un ajuste que no es una venta.
Los tres se ven igual en la columna.

**La pregunta de negocio: ¿un precio cero significa «gratis» o significa «no se sabe»?** La respuesta
cambia por completo qué hacer con el registro.

Variables que hay que mirar: `precio_unitario`, `id_cliente` y `cantidad` —el signo separa dos
situaciones distintas—.

In [41]:
precio_cero = limpio['precio_unitario'] == 0

print('Registros:', precio_cero.sum())
print()

Registros: 2510



In [42]:
pd.DataFrame({
    'con cliente': [
        (precio_cero & limpio['id_cliente'].notna() & (limpio['cantidad'] > 0)).sum(),
        (precio_cero & limpio['id_cliente'].notna() & (limpio['cantidad'] < 0)).sum(),
    ],
    'sin cliente': [
        (precio_cero & limpio['id_cliente'].isna() & (limpio['cantidad'] > 0)).sum(),
        (precio_cero & limpio['id_cliente'].isna() & (limpio['cantidad'] < 0)).sum(),
    ],
}, index=['cantidad positiva', 'cantidad negativa'])

,con cliente,sin cliente
cantidad positiva,40,1134
cantidad negativa,0,1336


### C · `pais`, donde `Unspecified` es un vacío disfrazado

`pais` no tiene ninguna celda vacía. Pero tiene categorías que **no nombran un país**.

**La pregunta de negocio: ¿es un país desconocido o es un país que no se registró?** Y una segunda,
que decide si hay algo que recuperar: **¿se puede deducir el país desde el cliente?**

Variables que hay que mirar: `pais` e `id_cliente`. La celda usa **`dropna()`** para quedarse solo
con los clientes que sí están identificados antes de buscarlos en el resto del archivo.

In [43]:
sospechosos = ['Unspecified', 'European Community']
marca = limpio['pais'].isin(sospechosos)

print(limpio.loc[marca, 'pais'].value_counts().to_string())
print()
print('Clientes distintos en esos registros:', limpio.loc[marca, 'id_cliente'].nunique())

pais
Unspecified           442
European Community     61

Clientes distintos en esos registros: 5


In [44]:
# ¿Se puede recuperar el país? Solo si esos mismos clientes aparecen con un país concreto
clientes = limpio.loc[marca, 'id_cliente'].dropna().unique()
otras_filas = limpio[limpio['id_cliente'].isin(clientes) & ~marca]

print('Registros de esos mismos clientes con un país concreto:', len(otras_filas))

Registros de esos mismos clientes con un país concreto: 0


### Cómo se aplica una decisión

Ya vieron cinco formas de corregir, y las cinco sirven acá. Elegir cuál corresponde **es** la
decisión; escribirla no debería costarles tiempo:

| Verbo | Se vio en | Se escribe así |
|:--|:--|:--|
| completar por regla | Decisión 1 | `limpio['col'] = limpio['col'].fillna(...)` |
| separar en dos columnas | Decisión 2 | `limpio['nueva'] = ...` |
| eliminar filas | Decisión 3 | `limpio = limpio[~mascara]` |
| normalizar en columna nueva | Decisión 4 | `limpio['col_norm'] = ...` |
| marcar con un indicador | Decisión 5 | `limpio['marca'] = mascara` |

Y la máscara de su alternativa **ya está calculada** por la celda de evidencia que acaban de
ejecutar: se llama `neg_sin_c` en la A, `precio_cero` en la B y `marca` en la C.

In [45]:
# SU SEGUNDA DECISIÓN
# la máscara ya existe: neg_sin_c (A), precio_cero (B) o marca (C)
# apliquen su decisión sobre `limpio` con uno de los cinco verbos de la tabla

In [46]:
# SU SEGUNDA LÍNEA DE BITÁCORA

---

# Lo que quedó hecho

Las dos mitades, juntas.

In [47]:
bitacora = pd.DataFrame(BITACORA)

print(f'Decisiones registradas: {len(BITACORA)} de {DECISIONES_ESPERADAS}')
print('Registros del archivo de trabajo:', len(limpio))

bitacora

Decisiones registradas: 5 de 7
Registros del archivo de trabajo: 536642


,variable,proceso generador,decisión,justificación,efecto esperado
0,descripcion_producto,registros que no son ventas y se cargaron sin ...,completar SOLO desde códigos con un único nomb...,"cuando el código conoce un solo nombre, ese no...",ninguna fila se pierde; quedan 421 sin descrip...
1,n_factura,el sistema marca la cancelación con un prefijo...,separar en n_documento y cancelada; se conserv...,dos hechos en una columna no se pueden contar ...,se puede contar cancelaciones sin parsear text...
2,(todas),cargas repetidas del mismo archivo; no se dist...,eliminar las 5268 repeticiones de FILA COMPLET...,una factura puede tener dos líneas del mismo p...,el archivo baja a 536642 filas; los hechos rep...
3,descripcion_producto,el campo del nombre se usó como cuaderno de no...,normalizar a minúsculas y sin espacios sobrant...,normalizar unifica variantes de la misma palab...,menos categorías duplicadas; los registros ano...
4,codigo_producto,la tabla de ventas se usa también para cargar ...,marcar las 2990 con la columna es_producto; no...,la mayoría son hechos contables reales y borra...,un análisis de productos puede filtrarlas; la ...


## El archivo corregido

La exportación **revisa primero que estén las siete decisiones**. Con menos, no escribe nada.

No es un capricho del cuaderno: un archivo corregido a medias parece terminado y no lo está, y quien
lo reciba no tiene cómo darse cuenta. La bitácora es lo que vuelve auditable al archivo, así que
salen juntos o no sale ninguno.

El dataset se escribe con **`to_parquet()`** y la bitácora con **`to_csv()`**: el primero conserva
los tipos para quien siga trabajando con el archivo, el segundo se abre en cualquier planilla.

In [48]:
faltan = DECISIONES_ESPERADAS - len(BITACORA)

if faltan > 0:
    print(f'NO se exportó nada: faltan {faltan} decisiones de las {DECISIONES_ESPERADAS}.')
    print()
    print('Para completar la Parte 3:')
    print('  1. Decisión 6 · elijan una opción para id_cliente, aplíquenla y anótenla.')
    print('  2. Decisión 7 · elijan A, B o C, apliquen su decisión y anótenla.')
    print()
    print('Después vuelvan a ejecutar esta celda.')
else:
    # la bitácora se arma ACÁ y no antes: si se reusara la de la celda de vista previa, una
    # decisión agregada después quedaría fuera del archivo y nadie lo notaría
    bitacora = pd.DataFrame(BITACORA)

    limpio.to_parquet('ventas_online_limpio.parquet', index=False)
    bitacora.to_csv('bitacora_limpieza.csv', index=False)
    print(f'Escritos, con las {len(BITACORA)} decisiones registradas:')
    print('  ventas_online_limpio.parquet · bitacora_limpieza.csv')

NO se exportó nada: faltan 2 decisiones de las 7.

Para completar la Parte 3:
  1. Decisión 6 · elijan una opción para id_cliente, aplíquenla y anótenla.
  2. Decisión 7 · elijan A, B o C, apliquen su decisión y anótenla.

Después vuelvan a ejecutar esta celda.


## Lo que queda para después

Tres cosas que **no** se hicieron hoy, a propósito:

- **Escalar, codificar y crear variables**: eso es la clase 5. Hoy se corrigió lo que estaba mal, no
  se transformó lo que ya estaba bien.
- **Separar entrenamiento y prueba**: es la clase 6.
- **Medir un modelo**: también la clase 6. Hoy no se midió nada.

Una regla que empieza a regir **desde ahora**, aunque su porqué llegue después: todo lo que
**aprende un parámetro de los datos** —el valor con que se imputa, el umbral con que se recorta— se
calcula **solo con los datos de entrenamiento**. Corregir un tipo, separar una columna o eliminar
una repetición identificada no aprende nada, así que puede hacerse sobre el archivo completo. Imputar
y recortar, no.

## Atribución de datos

- **Creador:** Chen, D. (2012)
- **Fuente:** [UCI Machine Learning Repository — *Online Retail II*, dataset 502](https://archive.ics.uci.edu/dataset/502/online+retail+ii)
- **Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)
- **Modificación:** las ocho columnas se renombraron al español en `snake_case`. Los valores no se
  tocaron: llegan en inglés, tal como están en el original.

---

# Segunda parte · representar lo que ya está bien

Hasta acá **corregimos** lo que estaba mal en el archivo y lo dejamos escrito en la bitácora. Lo que
sigue es la otra pregunta de la sesión: *¿esto está correcto pero en un formato que el modelo no
aprovecha?* El archivo es el mismo, ya con las decisiones aplicadas.


In [49]:
import os

import numpy as np
import pandas as pd

def num(x, dec=0):
    """Miles con punto y decimales con coma, como en el resto del material del curso."""
    return f'{x:,.{dec}f}'.replace(',', '@').replace('.', ',').replace('@', '.')

## Cargar datos

El archivo viene en **Parquet**, que guarda el tipo de cada columna, así que no hay que adivinar nada al
leerlo. Ya usamos `read_parquet` en la clase 4.

In [50]:
url_datasets = 'https://raw.githubusercontent.com/sebastiancontz/ust-introduccion-machine-learning-colab/main/ediciones/2026/datasets/'
ruta_datos = '../datasets/' if os.path.exists('../datasets') else url_datasets
ventas = pd.read_parquet(ruta_datos + 'ventas_online_limpio.parquet')

In [51]:
print(f'El archivo trae {num(len(ventas))} registros y {ventas.shape[1]} columnas.')

El archivo trae 536.642 registros y 14 columnas.


Cada **registro** es una **línea de factura**: un producto dentro de una factura. No es una venta completa y
no es un cliente.

### El filtro que fija la base, y se declara una sola vez

Nos quedamos con las líneas que son **ventas de producto efectivas**: que sean producto, que no estén
canceladas, y que tengan cantidad y precio positivos. Todas las cifras de este cuaderno salen de ahí.

<!-- contrato: accion=fijar-base -->

In [52]:
lineas = ventas[
    ventas["es_producto"] & ~ventas["cancelada"] & (ventas["cantidad"] > 0) & (ventas["precio_unitario"] > 0)
].copy()
lineas["monto_linea"] = lineas["cantidad"] * lineas["precio_unitario"]   # lo que vale esa línea

In [53]:
print(f'Quedan {num(len(lineas))} líneas de las {num(len(ventas))} originales.')

Quedan 522.504 líneas de las 536.642 originales.


## Y ahora cada registro va a ser una factura

Para lo que sigue necesitamos **un registro por factura**, no por línea. O sea que cambiamos la **unidad de
observación**, igual que en la clase 2, y por eso lo decimos en voz alta.

### `agg`: resumir cada grupo con la función que corresponda

`groupby` ya lo conocen: separa las filas en grupos. `agg` es lo que va después: recibe, por cada columna
nueva, **de dónde sale y cómo se resume**. El monto de una factura es la **suma** de sus líneas, su
número de líneas es un **conteo**, y su país es el **primero** —todas sus líneas comparten país—.

El **mes** no viene como columna: se saca de la fecha con el accesor `.dt`, que da acceso a las partes
de una fecha —año, mes, día, hora—. Es la familia de variables derivadas «componentes de una fecha».

<!-- contrato: accion=agregar-por-factura -->

In [54]:
facturas = lineas.groupby('n_documento').agg(
    monto_total=('monto_linea', 'sum'),
    n_lineas=('codigo_producto', 'size'),
    unidades=('cantidad', 'sum'),
    pais=('pais', 'first'),
    fecha=('fecha_factura', 'first'),
).reset_index()

In [55]:
facturas["mes"] = facturas["fecha"].dt.month   # el mes sale de la fecha, no viene como columna

In [56]:
print(f'De {num(len(lineas))} líneas quedaron {num(len(facturas))} facturas.')

De 522.504 líneas quedaron 19.773 facturas.


<p align="center"></p>

## Cómo vamos a medir

Necesitamos una forma de comparar dos maneras de escribir las mismas variables. Vamos a usar un **modelo
lineal** —el que ya conocen— como **instrumento de medición**, no como predictor: explica el monto de una
factura desde su número de líneas, sus unidades, su país y su mes.

Y lo vamos a reportar en **libras por factura**: la mitad de las facturas queda estimada con un error
menor a esa cifra. Nada de porcentajes abstractos.

### Dos detalles de la receta, y los dos importan

- El modelo trabaja sobre el **logaritmo** del monto, porque su distribución tiene una cola larguísima.
  Así que la predicción hay que **destransformarla** con `expm1` antes de compararla con libras reales.
- `log1p` calcula el logaritmo de **1 + x**, no de x. El logaritmo de cero no existe, y este archivo
  tiene facturas de una sola unidad.

### Dos nombres nuevos, y uno ya lo vieron en las slides

`make_pipeline` encadena pasos: le entregamos el preprocesador y el modelo, y devuelve un solo objeto
que aplica el primero y después el segundo. Es la forma corta de armar el **pipeline** del que hablamos
en clase.

Y una advertencia sobre la cifra: **se calcula sobre las mismas facturas con las que se ajustó el
modelo**. Sirve para comparar dos formas de escribir las mismas variables, que es lo único que
necesitamos hoy. Por qué eso no alcanza para saber si un modelo sirve es la clase 6.

In [57]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

def error_libras(preprocesador, columnas):
    """Mediana del error absoluto, en libras por factura."""
    modelo = make_pipeline(preprocesador, LinearRegression())
    modelo.fit(facturas[columnas], np.log1p(facturas["monto_total"]))
    prediccion = np.expm1(modelo.predict(facturas[columnas]))
    return float(np.median(np.abs(prediccion - facturas["monto_total"])))

In [58]:
print('Referencia de lectura: la factura mediana vale '
      f'{num(facturas["monto_total"].median())} libras.')

Referencia de lectura: la factura mediana vale 302 libras.


# Primer tiempo · el preprocesador mínimo

Cuatro columnas entran: dos numéricas y dos categóricas. Cada tipo necesita algo distinto.

### Las tres piezas que hacen falta

- **`OneHotEncoder`** convierte una columna de categorías en una columna por categoría. Con
  `handle_unknown='ignore'` no se cae si mañana aparece un país que no estaba.
- **`FunctionTransformer`** aplica una función cualquiera a las columnas que se le indiquen; acá le
  vamos a pasar `log1p`.
- **`ColumnTransformer`** es el que reparte: recibe una lista de tripletas —nombre, transformador,
  columnas— y reúne todas las salidas en una sola matriz.

### Y por qué las numéricas pasan por el logaritmo

Porque es la decisión que la teoría de hoy justificó, y conviene no ejecutarla a ciegas: `unidades`
tiene una cola larguísima, y su máximo son **80.995 unidades en una sola factura** — la misma que la
clase 4 discutió y decidió **conservar**, porque era una venta real con su devolución.

Transformar la variable es lo que desarma esa cola **sin borrar la venta**. Acá no lo volvemos a
demostrar: se aplica la decisión y se sigue.

In [59]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

columnas_numericas = ['n_lineas', 'unidades']
columnas_categoricas = ['pais', 'mes']

In [60]:
preprocesador = ColumnTransformer([
    ('num', FunctionTransformer(np.log1p), columnas_numericas),
    ('cat', OneHotEncoder(handle_unknown='ignore'), columnas_categoricas),
])

Hasta acá solo lo **declaramos**. Ahora lo ejecutamos sobre el archivo y miramos qué sale.

<!-- contrato: cifra=52 -->

In [61]:
matriz = preprocesador.fit_transform(facturas[columnas_numericas + columnas_categoricas])
print(f'Entraron {len(columnas_numericas + columnas_categoricas)} columnas y salieron {matriz.shape[1]}.')

Entraron 4 columnas y salieron 52.


### De dónde salen esas columnas

Las dos numéricas siguen siendo dos. Las categóricas son las que se multiplicaron: una columna por cada
valor distinto.

In [62]:
for columna in columnas_categoricas:
    print(f'{columna:8s} {facturas[columna].nunique():>3} valores distintos')

pais      38 valores distintos
mes       12 valores distintos


### Y casi todo lo que se agregó son ceros

In [63]:
# la matriz viene comprimida, sin guardar los ceros; la expandimos solo para contarlos
densa = matriz.toarray() if hasattr(matriz, 'toarray') else matriz
print(f'El {num(100 * (densa == 0).mean())} % de esa matriz son ceros.')

El 92 % de esa matriz son ceros.


<p align="center"></p>

### El error de este preprocesador

Guardemos la cifra, porque es la vara contra la que vamos a comparar todo lo demás.

In [64]:
error_base = error_libras(preprocesador, columnas_numericas + columnas_categoricas)
print(f'Error mediano: {num(error_base, 1)} libras por factura.')

Error mediano: 71,2 libras por factura.


# Segundo tiempo · dónde el one-hot deja de servir

Probemos con el **código de producto**. Y para eso hay que **volver a la tabla de líneas**, porque el
código de producto solo existe ahí: una factura tiene muchos productos, no un código.

In [65]:
n_productos = lineas["codigo_producto"].nunique()
print(f'Hay {num(n_productos)} productos distintos en {num(len(lineas))} líneas.')

Hay 3.900 productos distintos en 522.504 líneas.


### Antes de ejecutar nada, hagamos la cuenta

Si le aplicáramos el one-hot, tendríamos una columna por producto y un registro por línea.

In [66]:
celdas = len(lineas) * n_productos
gigas = celdas * 8 / 1e9   # 8 bytes por número en punto flotante
print(f'Serían {num(celdas / 1e6)} millones de celdas, '
      f'o {num(gigas, 1)} GB si se guardaran todas.')

Serían 2.038 millones de celdas, o 16,3 GB si se guardaran todas.


In [67]:
print(f'Y solo el {num(100 * len(lineas) / celdas, 3)} % sería distinto de cero.')

Y solo el 0,026 % sería distinto de cero.


### Qué concluimos

Dos cosas, y las dos importan:

1. Acá **la herramienta correcta no es esta**. Hay formas de guardar solo las celdas que no son cero, y
   con eso el archivo entra en memoria — pero el problema no era el espacio: el modelo seguiría teniendo
   que estimar un número por cada producto.
2. El código de producto **no puede entrar como una columna más** de la tabla por factura. Por eso hubo
   que agregar al principio. La agregación no fue una preferencia: fue la consecuencia de esto.

# Tercer tiempo · ahora ustedes

Toca **crear una variable nueva** que el archivo no trae, incorporarla al preprocesador y medirla.

## La candidata: el precio de catálogo medio de la canasta

La idea de negocio es simple: **una factura de productos caros vale más que una de productos baratos**,
aunque las dos tengan la misma cantidad de líneas y de unidades. Eso el archivo no lo dice directamente,
pero se puede construir.

Dos pasos:

1. Para cada producto, su **precio habitual**: la mediana de todos los precios a los que se vendió. Sale
   con el mismo `agg` de antes, solo que agrupando por producto en vez de por factura.
2. Para cada factura, el **promedio de esos precios habituales** entre los productos que lleva.

### `merge`: pegarle a cada línea un dato que vive en otra tabla

`merge` une dos tablas por una columna que comparten. Acá la tabla chica es el catálogo de precios
habituales, y la columna compartida es `codigo_producto`.

In [68]:
catalogo = lineas.groupby('codigo_producto').agg(
    precio_catalogo=('precio_unitario', 'median'),
).reset_index()

In [69]:
catalogo.head()

,codigo_producto,precio_catalogo
0,10002,0.85
1,10080,0.39
2,10120,0.21
3,10123C,0.65
4,10124A,0.42


Ahora se lo pegamos a cada línea, y de ahí lo resumimos por factura.

<!-- contrato: accion=pegar-catalogo -->

In [70]:
lineas_con_catalogo = lineas.merge(catalogo, on='codigo_producto', how='left')

In [71]:
por_factura = lineas_con_catalogo.groupby("n_documento")["precio_catalogo"].mean()
facturas["precio_catalogo_medio"] = facturas["n_documento"].map(por_factura)

In [72]:
print(f'La variable nueva va de {num(facturas["precio_catalogo_medio"].min(), 2)} '
      f'a {num(facturas["precio_catalogo_medio"].max())} libras, '
      f'con mediana {num(facturas["precio_catalogo_medio"].median(), 2)}.')

La variable nueva va de 0,04 a 165 libras, con mediana 2,78.


## Incorporarla al preprocesador

Va por la **ruta numérica**, junto a las otras dos, así que también pasa por el logaritmo.

In [73]:
columnas_numericas_con_precio = columnas_numericas + ['precio_catalogo_medio']
preprocesador_nuevo = ColumnTransformer([
    ('num', FunctionTransformer(np.log1p), columnas_numericas_con_precio),
    ('cat', OneHotEncoder(handle_unknown='ignore'), columnas_categoricas),
])

In [74]:
error_nuevo = error_libras(preprocesador_nuevo, columnas_numericas_con_precio + columnas_categoricas)
print(f'Error mediano: {num(error_nuevo, 1)} libras, contra {num(error_base, 1)} de antes.')

Error mediano: 43,8 libras, contra 71,2 de antes.


## Ahora la parte que se mira

La cifra bajó, y eso está bien. Pero **no es lo que se les pide justificar**.

Escriban, en una o dos frases:

- **de dónde sale** el valor de esta variable, o sea qué cuenta exactamente;
- **a qué hecho del negocio** corresponde, o sea qué decisión o comportamiento real refleja.

Sin mencionar cuánto bajó el error.

In [75]:
# escriban su justificación acá, entre las comillas
justificacion = """

"""
print(justificacion.strip() or 'Todavía sin escribir.')

Todavía sin escribir.


## Y si les queda tiempo

Prueben **otra** variable propia y compárenla. Dos ideas del archivo, las dos defendibles desde el
negocio:

- las **unidades por línea** de la factura, que distingue una compra al detalle de una al por mayor;
- la **hora** de la factura, que sale de `fecha_factura` igual que el mes.

La pregunta es la misma: qué cuenta, y qué hecho del negocio refleja.

## Atribución de datos

- **Creador:** Chen, D. (2012)
- **Fuente:** [UCI Machine Learning Repository — *Online Retail II*, dataset 502](https://archive.ics.uci.edu/dataset/502/online+retail+ii)
- **Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)
- **Modificación:** obra derivada en dos pasos. Las ocho columnas originales se renombraron al español
  en `snake_case`, sin traducir los valores; después se aplicaron las siete decisiones de limpieza de la
  clase 4, que eliminan filas repetidas y agregan seis columnas.